# Notebook 03 - Modelado\n\n**Responsable:** Integrante 2  \n**Bloque:** baseline + LightGBM + tuning + SHAP + evaluación\n\nEste notebook parte de los archivos procesados generados en `02_preprocessing.ipynb` y tiene cinco objetivos principales:\n\n1. cargar los splits `train / val / test` ya preparados\n2. entrenar uno o más baselines de referencia\n3. entrenar un modelo principal con **LightGBM**\n4. ajustar hiperparámetros sin contaminar el test set\n5. producir métricas, figuras y artefactos de explicabilidad con **SHAP**\n\n> Nota: el proyecto quedó orientado a **predicción binaria de deserción estudiantil**, consistente con la opción 1 definida por el equipo.

In [ ]:
from pathlib import Path\nimport json\nimport warnings\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport shap\n\nfrom lightgbm import LGBMClassifier\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.model_selection import PredefinedSplit, RandomizedSearchCV\n\nwarnings.filterwarnings('ignore')\nsns.set_theme(style='whitegrid')\nplt.rcParams['figure.figsize'] = (10, 6)\n\nRANDOM_STATE = 42

In [ ]:
ROOT = Path.cwd().resolve()\nif ROOT.name == 'notebooks':\n    ROOT = ROOT.parent\n\nDATA_DIR = ROOT / 'data' / 'processed'\nMODEL_DIR = ROOT / 'models'\nFIGURE_DIR = DATA_DIR\nCHECKPOINT_DIR = MODEL_DIR / 'checkpoints'\n\nMODEL_DIR.mkdir(parents=True, exist_ok=True)\nCHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)\n\nX_train = pd.read_csv(DATA_DIR / 'X_train.csv')\nX_val = pd.read_csv(DATA_DIR / 'X_val.csv')\nX_test = pd.read_csv(DATA_DIR / 'X_test.csv')\n\ny_train = pd.read_csv(DATA_DIR / 'y_train.csv')['Target_Binary']\ny_val = pd.read_csv(DATA_DIR / 'y_val.csv')['Target_Binary']\ny_test = pd.read_csv(DATA_DIR / 'y_test.csv')['Target_Binary']\n\nX_trainval = pd.concat([X_train, X_val], axis=0, ignore_index=True)\ny_trainval = pd.concat([y_train, y_val], axis=0, ignore_index=True)\n\nprint('Shapes:')\nprint(f'  X_train: {X_train.shape} | y_train: {y_train.shape}')\nprint(f'  X_val:   {X_val.shape} | y_val:   {y_val.shape}')\nprint(f'  X_test:  {X_test.shape} | y_test:  {y_test.shape}')\nprint(f'  Tasa de dropout en train: {y_train.mean():.3f}')\nprint(f'  Tasa de dropout en val:   {y_val.mean():.3f}')\nprint(f'  Tasa de dropout en test:  {y_test.mean():.3f}')

In [ ]:
def evaluate_binary_classifier(model, X, y, model_name, split_name):\n    y_pred = model.predict(X)\n\n    if hasattr(model, 'predict_proba'):\n        y_score = model.predict_proba(X)[:, 1]\n    else:\n        y_score = y_pred\n\n    return {\n        'model': model_name,\n        'split': split_name,\n        'accuracy': accuracy_score(y, y_pred),\n        'precision': precision_score(y, y_pred, zero_division=0),\n        'recall': recall_score(y, y_pred, zero_division=0),\n        'f1': f1_score(y, y_pred, zero_division=0),\n        'roc_auc': roc_auc_score(y, y_score),\n    }\n\n\ndef plot_confusion_matrix(y_true, y_pred, labels, title, save_path):\n    cm = confusion_matrix(y_true, y_pred)\n    plt.figure(figsize=(6, 5))\n    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)\n    plt.title(title)\n    plt.xlabel('Predicción')\n    plt.ylabel('Real')\n    plt.tight_layout()\n    plt.savefig(save_path, dpi=300, bbox_inches='tight')\n    plt.show()

## 1. Baselines\n\nSe usan dos referencias sencillas:\n\n- `dummy_prior`: establece el piso mínimo del problema.\n- `logreg_balanced`: baseline lineal más competitivo y fácil de interpretar.\n\nLa comparación inicial se hace sobre el **conjunto de validación** para no tocar todavía el test set.

In [ ]:
baseline_models = {\n    'dummy_prior': DummyClassifier(strategy='prior', random_state=RANDOM_STATE),\n    'logreg_balanced': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE),\n}\n\nvalidation_results = []\n\nfor model_name, model in baseline_models.items():\n    model.fit(X_train, y_train)\n    validation_results.append(\n        evaluate_binary_classifier(model, X_val, y_val, model_name=model_name, split_name='validation')\n    )\n\nvalidation_results_df = pd.DataFrame(validation_results).sort_values('f1', ascending=False)\nvalidation_results_df.round(4)

## 2. LightGBM base\n\nAquí se entrena una primera versión razonable del modelo principal.\n\nDecisiones heredadas del EDA y preprocessing:\n\n- el problema presenta **desbalance de clases**\n- se recomienda usar `class_weight='balanced'`\n- el modelo debe ser robusto en datos tabulares\n- LightGBM es adecuado para este tipo de variables

In [ ]:
lgbm_default = LGBMClassifier(\n    objective='binary',\n    class_weight='balanced',\n    random_state=RANDOM_STATE,\n    n_estimators=300,\n    learning_rate=0.05,\n    num_leaves=31,\n    subsample=0.9,\n    colsample_bytree=0.9,\n)\n\nlgbm_default.fit(X_train, y_train)\nvalidation_results.append(\n    evaluate_binary_classifier(lgbm_default, X_val, y_val, model_name='lightgbm_default', split_name='validation')\n)\n\nvalidation_results_df = pd.DataFrame(validation_results).sort_values('f1', ascending=False)\nvalidation_results_df.round(4)

## 3. Tuning con validación separada\n\nPara respetar la metodología del proyecto, el ajuste de hiperparámetros se hace usando **train + val** con un `PredefinedSplit`:\n\n- las filas originales de `train` se usan para entrenar cada candidato\n- las filas originales de `val` se usan para comparar candidatos\n- el **test set queda completamente reservado** para la evaluación final\n\nLa métrica principal de búsqueda será **F1**, porque el proyecto necesita un buen balance entre precision y recall en una clase minoritaria.

In [ ]:
validation_fold = np.concatenate([\n    np.full(len(X_train), -1),\n    np.zeros(len(X_val), dtype=int),\n])\n\npredefined_split = PredefinedSplit(test_fold=validation_fold)\n\nsearch_space = {\n    'n_estimators': [200, 300, 400, 500, 700],\n    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],\n    'num_leaves': [15, 31, 63, 127],\n    'max_depth': [-1, 5, 8, 12],\n    'min_child_samples': [10, 20, 40, 60],\n    'subsample': [0.7, 0.8, 0.9, 1.0],\n    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],\n    'reg_alpha': [0.0, 0.1, 0.5, 1.0],\n    'reg_lambda': [0.0, 0.1, 0.5, 1.0],\n}\n\ntuner = RandomizedSearchCV(\n    estimator=LGBMClassifier(\n        objective='binary',\n        class_weight='balanced',\n        random_state=RANDOM_STATE,\n    ),\n    param_distributions=search_space,\n    n_iter=20,\n    scoring='f1',\n    cv=predefined_split,\n    random_state=RANDOM_STATE,\n    n_jobs=-1,\n    refit=True,\n    verbose=1,\n)\n\ntuner.fit(X_trainval, y_trainval)\nbest_lgbm = tuner.best_estimator_\n\nprint('Mejores hiperparámetros encontrados:')\nprint(json.dumps(tuner.best_params_, indent=2, ensure_ascii=False))\nprint(f"\nMejor F1 en validación: {tuner.best_score_:.4f}")

## 4. Evaluación final en test\n\nLa comparación final se hace de forma justa:\n\n- se reentrenan los baselines en `train + val`\n- el modelo tuneado ya queda refiteado sobre `train + val` por `RandomizedSearchCV`\n- se compara todo una sola vez sobre `test`

In [ ]:
validation_results_df.to_csv(MODEL_DIR / 'metrics_validation_candidates.csv', index=False)\n\nfinal_models = {\n    'dummy_prior': DummyClassifier(strategy='prior', random_state=RANDOM_STATE).fit(X_trainval, y_trainval),\n    'logreg_balanced': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE).fit(X_trainval, y_trainval),\n    'lightgbm_tuned': best_lgbm,\n}\n\ntest_results = []\nfor model_name, model in final_models.items():\n    test_results.append(\n        evaluate_binary_classifier(model, X_test, y_test, model_name=model_name, split_name='test')\n    )\n\ntest_results_df = pd.DataFrame(test_results).sort_values('f1', ascending=False)\ntest_results_df.to_csv(MODEL_DIR / 'metrics_test_models.csv', index=False)\npd.DataFrame(tuner.cv_results_).sort_values('rank_test_score').to_csv(MODEL_DIR / 'lightgbm_tuning_results.csv', index=False)\n\nwith open(MODEL_DIR / 'lightgbm_best_params.json', 'w', encoding='utf-8') as f:\n    json.dump(tuner.best_params_, f, indent=2, ensure_ascii=False)\n\njoblib.dump(best_lgbm, CHECKPOINT_DIR / 'lightgbm_dropout_model.pkl')\n\ntest_results_df.round(4)

In [ ]:
comparison_df = test_results_df.melt(\n    id_vars='model',\n    value_vars=['f1', 'roc_auc', 'recall'],\n    var_name='metric',\n    value_name='score',\n)\n\nplt.figure(figsize=(10, 6))\nsns.barplot(data=comparison_df, x='metric', y='score', hue='model')\nplt.ylim(0, 1)\nplt.title('Comparación final de modelos en test')\nplt.tight_layout()\nplt.savefig(FIGURE_DIR / 'fig8_model_comparison.png', dpi=300, bbox_inches='tight')\nplt.show()\n\nbest_test_model = final_models['lightgbm_tuned']\nbest_test_pred = best_test_model.predict(X_test)\nplot_confusion_matrix(\n    y_true=y_test,\n    y_pred=best_test_pred,\n    labels=['No dropout', 'Dropout'],\n    title='Matriz de confusión - LightGBM tuneado',\n    save_path=FIGURE_DIR / 'fig9_confusion_matrix.png',\n)

## 5. SHAP para explicabilidad\n\nEste bloque produce una vista compacta de las variables con mayor impacto en el modelo final.\n\nLa idea es que estos resultados alimenten luego el bloque del Integrante 3, donde el agente conversacional transformará la salida del modelo en explicaciones en lenguaje natural.

In [ ]:
sample_size = min(300, len(X_test))\nX_shap = X_test.sample(sample_size, random_state=RANDOM_STATE)\n\nexplainer = shap.TreeExplainer(best_lgbm)\nshap_values = explainer.shap_values(X_shap)\n\nif isinstance(shap_values, list):\n    shap_array = shap_values[1]\nelse:\n    shap_array = shap_values\n\nmean_abs_shap = pd.Series(np.abs(shap_array).mean(axis=0), index=X_shap.columns)\ntop_shap_features = mean_abs_shap.sort_values(ascending=False).head(15)\ntop_shap_features.to_csv(MODEL_DIR / 'shap_top_features.csv', header=['mean_abs_shap'])\n\nshap.summary_plot(shap_array, X_shap, plot_type='bar', max_display=12, show=False)\nplt.tight_layout()\nplt.savefig(FIGURE_DIR / 'fig10_shap_summary.png', dpi=300, bbox_inches='tight')\nplt.show()\n\ntop_shap_features

## Entregables esperados de este notebook\n\nAl correrlo deberían quedar generados estos artefactos:\n\n- `models/metrics_validation_candidates.csv`\n- `models/metrics_test_models.csv`\n- `models/lightgbm_best_params.json`\n- `models/lightgbm_tuning_results.csv`\n- `models/shap_top_features.csv`\n- `models/checkpoints/lightgbm_dropout_model.pkl`\n- `data/processed/fig8_model_comparison.png`\n- `data/processed/fig9_confusion_matrix.png`\n- `data/processed/fig10_shap_summary.png`\n\nCon eso ya tienes base sólida para tu parte del informe:\n\n- **Sección 3:** Arquitectura del sistema\n- **Sección 4:** Metodología\n- soporte visual para resultados y explicabilidad